# Inversion run plots

All the plotting logic for the activation-inversion runs lives in this notebook: edit a cell to slice step ranges,
pick lines, or recolor, then re-run that cell.

**Data sources.** Nothing is imported from the repo. Every number comes from three files that `inversion/train.py`
writes into each run's directory, `RUNS_DIR/<run>/`:

| file | one row = |
|---|---|
| `config.json` | the run's settings (init, source, batch, LR, logging cadence, git commit) |
| `train_log.jsonl` | the **mean over `log_every` training steps** (`step` = the last of them): loss, token accuracy, and per sample source (`loss_s1`, `loss_s2`, …) which model's activations those samples came from |
| `val_log.jsonl` | **one val pass on one val source** (every `val_every` steps): loss, token accuracy, exact match, and the first 4 fixed val windows with the inverter's output |

Cells 2–4 print exactly which files were read and each run's settings.

In [ ]:
# Everything this notebook uses. Nothing comes from the repo (no `inversion.*` imports).
import json                      # parse config.json and the .jsonl logs
from pathlib import Path         # file paths

import matplotlib.pyplot as plt  # plotting
import pandas as pd              # tables, for slicing and filtering

pd.set_option("display.max_colwidth", None)

In [ ]:
# Where the runs live: relative to this notebook (inversion/), so ../runs/b128 is <repo>/runs/b128.
RUNS_DIR = Path("../runs/b128").resolve()
RUNS = ["s1_inv_s1_L6", "s1_inv_s2_L6", "rand_inv_s2_L6", "s1_inv_mixs12_L6", "rand_inv_mixs12_L6"]
print("RUNS_DIR =", RUNS_DIR)

In [ ]:
# Read the three files of every run. These dicts hold the raw rows; the tables below are built only from them.
def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

configs = {run: json.loads((RUNS_DIR / run / "config.json").read_text()) for run in RUNS}
raw_train = {run: read_jsonl(RUNS_DIR / run / "train_log.jsonl") for run in RUNS}
raw_val = {run: read_jsonl(RUNS_DIR / run / "val_log.jsonl") for run in RUNS}

# Which files were read, how many rows each has, and how far each run has got.
files = pd.DataFrame(
    [{"run": run, "file": str(RUNS_DIR / run / name), "rows": len(rows), "last_step": rows[-1]["step"]}
     for run in RUNS for name, rows in [("train_log.jsonl", raw_train[run]), ("val_log.jsonl", raw_val[run])]]
)
display(files)

# The settings each run was trained with (from its config.json).
SETTINGS = ["init", "source", "layer", "batch", "lr", "warmup", "log_every", "val_every", "n_val", "git_commit"]
settings = pd.DataFrame([{"run": run, **{k: configs[run].get(k) for k in SETTINGS}} for run in RUNS]).set_index("run")
display(settings)

In [ ]:
# Tidy tables: one row per line-point, so any slice is a plain pandas filter.

# train: one row per (run, step, sample_source). Mixed runs have two sample sources (s1 and s2) per log row.
rows = []
for run in RUNS:
    for r in raw_train[run]:
        for src in [k[len("loss_"):] for k in r if k.startswith("loss_")]:
            if r[f"n_{src}"] == 0:  # no samples of this source in these steps
                continue
            rows.append({"run": run, "init": configs[run]["init"], "trained_on": configs[run]["source"],
                         "step": r["step"], "sample_source": src, "loss": r[f"loss_{src}"],
                         "tok_acc": r[f"tok_acc_{src}"], "n": r[f"n_{src}"]})
train = pd.DataFrame(rows)

# val: one row per (run, step, val_source). Every run is evaluated on both s1 and s2, whatever it trained on.
val = pd.DataFrame(
    [{"run": run, "init": configs[run]["init"], "trained_on": configs[run]["source"], "step": r["step"],
      "val_source": r["source"], "loss": r["loss"], "tok_acc": r["tok_acc"], "exact_match": r["exact_match"]}
     for run in RUNS for r in raw_val[run]]
)

# val_examples: the same 4 fixed val windows at every val pass, with each run's output.
val_examples = pd.DataFrame(
    [{"run": run, "step": r["step"], "val_source": r["source"], "idx": ex["idx"], "n_correct": ex["n_correct"],
      "input_text": ex["input_text"], "output_text": ex["output_text"]}
     for run in RUNS for r in raw_val[run] for ex in r.get("examples", [])]
)

display(train.head(3), val.head(3), val_examples.head(3))

In [ ]:
# Style: edit here to recolor or relabel. Blue / orange / aqua-green are checked as a colorblind-safe set.
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
RED, YELLOW, PURPLE, NAVY = "#d62a2a", "#d69f2a", "#9f2ad6", "#2a4ed6"
INK, MUTED, GRID = "#2b2b29", "#6b6a64", "#e6e5e0"

# One entry per line: (run, source) -> color. "source" is the sample source for train plots, the val source for val.
COLOR = {
    ("rand_inv_s2_L6", "s2"): RED,
    ("s1_inv_s2_L6", "s2"): ORANGE,
    ("s1_inv_s1_L6", "s1"): BLUE,
    ("rand_inv_mixs12_L6", "s1"): BLUE,
    ("rand_inv_mixs12_L6", "s2"): ORANGE,
    ("s1_inv_mixs12_L6", "s2"): ORANGE,
    ("s1_inv_mixs12_L6", "s1"): BLUE,
}
LINESTYLE = {"random": "--", "s1": "-"}  # by inverter init: dashed random, solid s1-pretrained

# Legend labels for the train plots (val plots add " on <source>").
LABEL = {
    ("rand_inv_s2_L6", "s2"): "rand_inv_s2",
    ("s1_inv_s2_L6", "s2"): "s1_inv_s2",
    ("s1_inv_s1_L6", "s1"): "s1_inv_s1",
    ("rand_inv_mixs12_L6", "s1"): "rand_inv_mixs12 (s1 samples)",
    ("rand_inv_mixs12_L6", "s2"): "rand_inv_mixs12 (s2 samples)",
    ("s1_inv_mixs12_L6", "s2"): "s1_inv_mixs12 (s2 samples)",
    ("s1_inv_mixs12_L6", "s1"): "s1_inv_mixs12 (s1 samples)",
}

# Panels: title -> the (run, source) lines in it, in legend order.
PANELS = {
    "A)  single-source runs": [("rand_inv_s2_L6", "s2"), ("s1_inv_s2_L6", "s2"), ("s1_inv_s1_L6", "s1")],
    "B)  mixed runs (loss split by sample source)": [
        ("rand_inv_mixs12_L6", "s1"), ("rand_inv_mixs12_L6", "s2"),
        ("s1_inv_mixs12_L6", "s2"), ("s1_inv_mixs12_L6", "s1"),
    ],
}

In [ ]:
def plot_panels(df, y, panels, source_col, step_range=None, title="", ylabel="", labels=LABEL, ylim=None,
                figsize=(10, 9), ema_span=None):
    """One subplot per panel, stacked; each (run, source) is one line drawn through its logged points.

    df: `train` (source_col="sample_source") or `val` (source_col="val_source"); y: the column to plot.
    When `ema_span` is set, plot an EMA line and show the raw points as lighter dots.
    The y axis is linear and shared; by default it runs from 0 to just above the highest point drawn.
    """
    if step_range is not None:
        df = df[df.step.between(*step_range)]
    fig, axes = plt.subplots(len(panels), 1, figsize=figsize, sharex=True, sharey=True, squeeze=False)
    axes = axes[:, 0]
    for ax, (panel_title, lines) in zip(axes, panels.items()):
        for run, src in lines:
            d = df[(df.run == run) & (df[source_col] == src)].sort_values("step")
            style = LINESTYLE.get(configs[run]["init"], "-")
            color = COLOR[run, src]
            if ema_span is None:
                ax.plot(d.step, d[y], style, color=color, lw=1,
                        marker="o", ms=2.5, label=labels[run, src])
            else:
                ax.plot(d.step, d[y], linestyle="none", marker="o", ms=2.5,
                        color=color, alpha=0.15)
                ax.plot(d.step, d[y].ewm(span=ema_span, adjust=False).mean(), style,
                        color=color, lw=1.5, label=labels[run, src])
        ax.set_title(panel_title, loc="left", color=INK, fontsize=11)
        ax.set_ylabel(ylabel, color=INK)
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9, labelcolor=INK,
                  handlelength=3)
        ax.grid(True, color=GRID, lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color(MUTED)
        ax.tick_params(colors=MUTED)
    # Set the shared y range after every panel is drawn, so no panel clips another's lines.
    y_max = max((max(line.get_ydata(), default=0) for ax in axes for line in ax.get_lines()), default=1)
    axes[0].set_ylim(*(ylim or (0, y_max * 1.05)))
    if step_range is not None:
        axes[-1].set_xlim(*step_range)
    axes[-1].set_xlabel("training steps", color=INK)
    fig.suptitle(title, color=INK, fontsize=14)
    fig.tight_layout()
    return fig


# Shared settings for the titles (every run in RUNS_DIR was trained with the same ones).
LOG_EVERY, BATCH = settings.log_every.iloc[0], settings.batch.iloc[0]
VAL_EVERY, N_VAL = settings.val_every.iloc[0], settings.n_val.iloc[0]

## Train loss

In [ ]:
STEP_RANGE = (0, 2500)
fig = plot_panels(train, "loss", PANELS, "sample_source", STEP_RANGE,
                  title=f"Train loss, every {LOG_EVERY} steps. Batch size {BATCH}.", ylabel="cross-entropy (nats)",)
# fig.savefig(RUNS_DIR / "train_loss_AB_0-2500.png", dpi=150)

In [ ]:
STEP_RANGE = (1200, 10000)
fig = plot_panels(train, "loss", PANELS, "sample_source", STEP_RANGE,
                  title=f"Train loss, every {LOG_EVERY} steps. Batch size {BATCH}.", ylabel="cross-entropy (nats)", ema_span=50)
# fig.savefig(RUNS_DIR / "train_loss_AB_2500-5000.png", dpi=150)

## Validation

Same panels, but each line is the run evaluated on the val windows of the source it trained on. Every run is also
evaluated on the other seed (~13 nats, ~0 accuracy); to plot that, add e.g. `("s1_inv_s1_L6", "s2")` to a panel along
with a `COLOR` and `VAL_LABEL` entry.

In [ ]:
VAL_LABEL = {key: f"{LABEL[key].split(' (')[0]} on {key[1]}" for key in LABEL}
VAL_PANELS = {"A)  single-source runs": PANELS["A)  single-source runs"],
              "B)  mixed runs (evaluated on each source)": PANELS["B)  mixed runs (loss split by sample source)"]}
STEP_RANGE = None  # all of it; e.g. (0, 5000)
fig = plot_panels(val, "loss", VAL_PANELS, "val_source", STEP_RANGE, labels=VAL_LABEL,
                  title=f"Val loss, every {VAL_EVERY} steps, {N_VAL} windows per source.", ylabel="cross-entropy (nats)")
# fig.savefig(RUNS_DIR / "val_loss_AB.png", dpi=150)

In [ ]:
fig = plot_panels(val, "tok_acc", VAL_PANELS, "val_source", STEP_RANGE, labels=VAL_LABEL, ylim=(0, 1),
                  title=f"Val token accuracy, every {VAL_EVERY} steps, {N_VAL} windows per source.",
                  ylabel="token accuracy")
# fig.savefig(RUNS_DIR / "val_tok_acc_AB.png", dpi=150)

## Val examples

The first 4 val windows are the same at every val pass. This table shows each run's output for them at one step (the
latest step that every run has reached, unless you set `STEP`).

In [ ]:
common_steps = set.intersection(*(set(g.step) for _, g in val_examples.groupby("run")))
STEP = max(common_steps)
VAL_SOURCE = "s2"
ex = val_examples[(val_examples.step == STEP) & (val_examples.val_source == VAL_SOURCE)]
table = ex.pivot(index="idx", columns="run", values="output_text")[RUNS]
table.insert(0, "input", ex.groupby("idx").input_text.first())
print(f"step {STEP}, val source {VAL_SOURCE}")
display(table)